In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from shapely import wkb
from joblib import Memory 
from itertools import cycle

# custom code
#from utilities import run_sql

In [1]:
# Query the database to receive the outlines of administrative regions in Germany. (Europe available!)
df_trips = run_sql('trips')#, params={'limit': '500000'}) # double percent gets interpreted as single percent in SQL Query
df_trips['energy'] = df_trips['distance'] / 1000 # Verbruach von 1kWh pro km
df_trips

NameError: name 'run_sql' is not defined

In [3]:
df_trips

NameError: name 'df_trips' is not defined

In [ ]:
df_fleet = run_sql('fleet', index_col='vehicle_id') # double percent gets interpreted as single percent in SQL Query

df_fleet

In [ ]:
df_grouped = df_trips.groupby(['tour_id', 'vehicle_id']).agg(
    {'energy': 'sum',
      'distance': 'mean'}).reset_index()
df_grouped

In [4]:
df_fleet.fleet_test_id.value_counts()

NameError: name 'df_fleet' is not defined

In [ ]:
SPEDITIONEN = {1: 'Metzger',
               2: 'Wenner',
               3: 'Märker',
               4: 'Schwarz',
               5: 'Elflein',
               6: 'Schmid'}

for BATT_CAP in [300,400,500]:
  df_grouped['public'] = (df_grouped['energy'] - BATT_CAP).apply(lambda x: max(0, x))
  df_grouped['private'] = df_grouped['energy'].apply(lambda energy: min(energy, BATT_CAP))
  df_grouped_cluster = df_grouped.groupby('vehicle_id').agg(
      {'private': 'sum',
        'public': 'sum'})#.reset_index()
  df_grouped_cluster = df_grouped_cluster.join(df_fleet, on='vehicle_id', how='inner')
  if BATT_CAP == 300:
    df_grouped_cluster_300 = df_grouped_cluster
    unique_fleet_ids = df_grouped_cluster['fleet_test_id'].unique()
    markers = ['o', 's', '^', 'D', 'p', '*', 'H', '+', 'x', '|', '_']  # List of possible markers
        
    for fleet_id, marker in zip(unique_fleet_ids, markers):
          fleet_data = df_grouped_cluster[df_grouped_cluster['fleet_test_id'] == fleet_id]
          plt.scatter(fleet_data['private'], fleet_data['public'], 
                      label=f'Fleet {SPEDITIONEN[fleet_id]}', marker=marker)


  if BATT_CAP == 500:
    df_grouped_cluster_500 = df_grouped_cluster
  print(f"Capcity {BATT_CAP}, total public charging: {df_grouped['public'].sum()}, total private charging: {df_grouped['private'].sum()}")

plt.legend()

plt.xlabel("Geladene Energie am Depot (kWh)")
plt.ylabel("Geladene Energie an öffentlichen Ladestationen (kWh)")
plt.title("Ladeverhalten von Speditionen bei verschiedenen Batteriekapazitäten")



# Loop over the values of df_grouped_cluster_300 and df_grouped_cluster_500
for i in range(len(df_grouped_cluster_300)):
    plt.arrow(
        df_grouped_cluster_300['private'].iloc[i],
        df_grouped_cluster_300['public'].iloc[i],
        df_grouped_cluster_500['private'].iloc[i] - df_grouped_cluster_300['private'].iloc[i],
        df_grouped_cluster_500['public'].iloc[i] - df_grouped_cluster_300['public'].iloc[i],
        head_width=500, head_length=1000, fc='blue', ec='blue', length_includes_head=True, alpha=0.2
    )

plt.show()
#shape='full', color='b', lw=1, length_includes_head=True, 
#zorder=0, head_length=3., head_width=1.5


In [ ]:
display(df_grouped_cluster_300['private'])
display(df_grouped_cluster_300['public'])
display(df_grouped_cluster_500['private'])
display(df_grouped_cluster_300['private'],)

In [ ]:
df_grouped

sns.pairplot(df_grouped[['private', 'public', 'energy', 'distance']])


In [ ]:
sns.pairplot(df_trips[['distance', 'energy', 'avg_speed']].sample(1000))